# Column-Family Stores: Partitions, Rows, and Time
Cassandra (CQL) + Python lab

Run the cells in order, top to bottom. Each cell is one question (or one part of a question) from the handout.

Before running the first code cell: make sure Docker Desktop is running, and start Cassandra with:

```
docker run -d -p 9042:9042 cassandra:5.0
```

Give it 1-2 minutes to finish starting, then run the setup cell below.

## Setup: connect to Cassandra

In [6]:
from datetime import datetime
from cassandra.cluster import Cluster

cluster = Cluster(["127.0.0.1"], port=9042)
session = cluster.connect()
session.execute("CREATE KEYSPACE IF NOT EXISTS hospital WITH replication = "
                "{'class': 'SimpleStrategy', 'replication_factor': 1}")
session.set_keyspace("hospital")

def clean(value):
    if isinstance(value, datetime):
        return value.strftime("%Y-%m-%d %H:%M:%S")
    return value

def run(cql):
    rows = session.execute(cql, timeout=60)
    return [[clean(v) for v in row] for row in rows]


## Part A: Create

### Question 1

In [7]:
run("""
CREATE TABLE vitals_by_patient (
    patient_id text,
    reading_time timestamp,
    heart_rate int,
    spo2 int,
    PRIMARY KEY ((patient_id), reading_time)
) WITH CLUSTERING ORDER BY (reading_time DESC)
""")

print(run("SELECT column_name, kind, clustering_order "
           "FROM system_schema.columns "
           "WHERE keyspace_name = 'hospital' "
           "AND table_name = 'vitals_by_patient'"))


[['heart_rate', 'regular', 'none'], ['patient_id', 'partition_key', 'none'], ['reading_time', 'clustering', 'desc'], ['spo2', 'regular', 'none']]


### Question 2

In [8]:
run("INSERT INTO vitals_by_patient (patient_id, reading_time, heart_rate, spo2) "
    "VALUES ('P-40231', '2026-09-14 09:00:00+0000', 78, 97)")
run("INSERT INTO vitals_by_patient (patient_id, reading_time, heart_rate, spo2) "
    "VALUES ('P-40231', '2026-09-14 09:00:02+0000', 80, 96)")
run("INSERT INTO vitals_by_patient (patient_id, reading_time, heart_rate, spo2) "
    "VALUES ('P-40231', '2026-09-14 09:00:04+0000', 79, 97)")

print(run("SELECT reading_time, heart_rate, spo2 FROM vitals_by_patient "
           "WHERE patient_id = 'P-40231'"))


[['2026-09-14 09:00:04', 79, 97], ['2026-09-14 09:00:02', 80, 96], ['2026-09-14 09:00:00', 78, 97]]


### Question 3

In [9]:
run("""
CREATE TABLE ledger_by_account (
    account_id text,
    entry_time timestamp,
    amount int,
    description text,
    PRIMARY KEY ((account_id), entry_time)
) WITH CLUSTERING ORDER BY (entry_time DESC)
""")

run("INSERT INTO ledger_by_account (account_id, entry_time, amount, description) "
    "VALUES ('acct-771', '2026-09-14 08:55:00+0000', 120000, 'Opening deposit')")
run("INSERT INTO ledger_by_account (account_id, entry_time, amount, description) "
    "VALUES ('acct-771', '2026-09-14 09:14:00+0000', -2500, 'ATM withdrawal')")
run("INSERT INTO ledger_by_account (account_id, entry_time, amount, description) "
    "VALUES ('acct-771', '2026-09-14 11:30:00+0000', -1200, 'Mobile top-up')")

print(run("SELECT entry_time, amount, description FROM ledger_by_account "
           "WHERE account_id = 'acct-771'"))


[['2026-09-14 11:30:00', -1200, 'Mobile top-up'], ['2026-09-14 09:14:00', -2500, 'ATM withdrawal'], ['2026-09-14 08:55:00', 120000, 'Opening deposit']]


## Part B: Read

### Question 4
Loads the full dataset and replaces the practice rows from Part A.

In [10]:
from load_cassandra import load
load(session)

print(run("SELECT COUNT(*) FROM vitals_by_patient WHERE patient_id = 'P-40231'"))
print(run("SELECT COUNT(*) FROM vitals_by_patient WHERE patient_id = 'P-40232'"))
print(run("SELECT COUNT(*) FROM vitals_by_patient WHERE patient_id = 'P-40233'"))


[[10]]
[[10]]
[[10]]


### Question 5

In [11]:
print(run("SELECT reading_time, heart_rate, spo2 FROM vitals_by_patient "
           "WHERE patient_id = 'P-40232' LIMIT 1"))


[['2026-09-14 09:00:18', 89, 94]]


### Question 6

In [12]:
print(run("SELECT reading_time, heart_rate, spo2 FROM vitals_by_patient "
           "WHERE patient_id = 'P-40231' "
           "AND reading_time >= '2026-09-14 09:00:06+0000' "
           "AND reading_time <= '2026-09-14 09:00:12+0000'"))


[['2026-09-14 09:00:12', 77, 96], ['2026-09-14 09:00:10', 78, 97], ['2026-09-14 09:00:08', 80, 98], ['2026-09-14 09:00:06', 81, 97]]


### Question 7

In [13]:
print(run("SELECT reading_time, heart_rate, spo2 FROM vitals_by_patient "
           "WHERE patient_id = 'P-40233' "
           "ORDER BY reading_time ASC LIMIT 3"))


[['2026-09-14 09:00:00', 70, 96], ['2026-09-14 09:00:02', 72, 95], ['2026-09-14 09:00:04', 71, 94]]


### Question 8

In [14]:
# (a)
print(run("SELECT MIN(spo2), MAX(heart_rate) FROM vitals_by_patient "
           "WHERE patient_id = 'P-40232'"))

# (b)
print(run("SELECT SUM(amount) FROM ledger_by_account WHERE account_id = 'acct-771'"))
print(run("SELECT SUM(amount) FROM ledger_by_account WHERE account_id = 'acct-772'"))


[[88, 99]]
[[150500]]
[[33000]]


## Part C: Update

### Question 9

In [15]:
run("INSERT INTO vitals_by_patient (patient_id, reading_time, heart_rate, spo2) "
    "VALUES ('P-40231', '2026-09-14 09:00:04+0000', 85, 97)")

print(run("SELECT COUNT(*) FROM vitals_by_patient WHERE patient_id = 'P-40231'"))
print(run("SELECT reading_time, heart_rate, spo2 FROM vitals_by_patient "
           "WHERE patient_id = 'P-40231' AND reading_time = '2026-09-14 09:00:04+0000'"))


[[10]]
[['2026-09-14 09:00:04', 85, 97]]


### Question 10

In [16]:
# (a)
run("UPDATE vitals_by_patient SET spo2 = 95 "
    "WHERE patient_id = 'P-40231' AND reading_time = '2026-09-14 09:00:06+0000'")
print(run("SELECT spo2 FROM vitals_by_patient "
           "WHERE patient_id = 'P-40231' AND reading_time = '2026-09-14 09:00:06+0000'"))

# (b)
run("UPDATE vitals_by_patient SET heart_rate = 70 "
    "WHERE patient_id = 'P-40231' AND reading_time = '2026-09-14 09:01:00+0000'")

print(run("SELECT COUNT(*) FROM vitals_by_patient WHERE patient_id = 'P-40231'"))
print(run("SELECT reading_time, heart_rate, spo2 FROM vitals_by_patient "
           "WHERE patient_id = 'P-40231' AND reading_time = '2026-09-14 09:01:00+0000'"))


[[95]]
[[11]]
[['2026-09-14 09:01:00', 70, None]]


## Part D: Delete

### Question 11

In [17]:
run("DELETE FROM vitals_by_patient "
    "WHERE patient_id = 'P-40231' AND reading_time = '2026-09-14 09:01:00+0000'")

print(run("SELECT COUNT(*) FROM vitals_by_patient WHERE patient_id = 'P-40231'"))
print(run("SELECT reading_time, heart_rate, spo2 FROM vitals_by_patient "
           "WHERE patient_id = 'P-40231' AND reading_time = '2026-09-14 09:01:00+0000'"))


[[10]]
[]


### Question 12

In [18]:
run("DELETE FROM vitals_by_patient WHERE patient_id = 'P-40231'")

print(run("SELECT COUNT(*) FROM vitals_by_patient WHERE patient_id = 'P-40231'"))


[[0]]


## Part E: Design and limits

### Question 13

In [19]:
# (a)
try:
    run("SELECT patient_id, reading_time, spo2 FROM vitals_by_patient WHERE spo2 < 90")
except Exception as error:
    print(type(error).__name__)

# (b)
rows = run("SELECT patient_id, reading_time, spo2 FROM vitals_by_patient "
           "WHERE spo2 < 90 ALLOW FILTERING")
print(sorted(rows))


InvalidRequest
[['P-40232', '2026-09-14 09:00:08', 89], ['P-40232', '2026-09-14 09:00:10', 88], ['P-40233', '2026-09-14 09:00:12', 89], ['P-40233', '2026-09-14 09:00:14', 87]]


### Question 14

In [20]:
# (a)
run("""
CREATE TABLE low_spo2_by_ward (
    ward text,
    reading_time timestamp,
    patient_id text,
    spo2 int,
    PRIMARY KEY ((ward), reading_time, patient_id)
) WITH CLUSTERING ORDER BY (reading_time DESC, patient_id ASC)
""")

# (b)
run("INSERT INTO low_spo2_by_ward (ward, reading_time, patient_id, spo2) "
    "VALUES ('ICU-A', '2026-09-14 09:00:08+0000', 'P-40232', 89)")
run("INSERT INTO low_spo2_by_ward (ward, reading_time, patient_id, spo2) "
    "VALUES ('ICU-A', '2026-09-14 09:00:10+0000', 'P-40232', 88)")
run("INSERT INTO low_spo2_by_ward (ward, reading_time, patient_id, spo2) "
    "VALUES ('ICU-B', '2026-09-14 09:00:12+0000', 'P-40233', 89)")
run("INSERT INTO low_spo2_by_ward (ward, reading_time, patient_id, spo2) "
    "VALUES ('ICU-B', '2026-09-14 09:00:14+0000', 'P-40233', 87)")

# (c)
print(run("SELECT reading_time, patient_id, spo2 FROM low_spo2_by_ward WHERE ward = 'ICU-A'"))


[['2026-09-14 09:00:10', 'P-40232', 88], ['2026-09-14 09:00:08', 'P-40232', 89]]


### Question 15
Written answer (no code) — edit this cell with your own wording:

**(a)** Store every heart rate/oxygen reading from every ICU patient, continuously, for months. -> *Good fit.* This is exactly what a column-family store is built for: nonstop appends into one partition per patient, read back as a time-ordered slice.

**(b)** Find the average oxygen level of all patients, month by month, over the last year. -> *Not a good fit.* This aggregates across every partition, which Cassandra cannot do efficiently. A column-oriented analytical store (e.g. BigQuery/a data warehouse) is built for scanning and averaging one column across all rows.

**(c)** Find every account that received money from a suspicious account. -> *Not a good fit.* This is a relationship/traversal question (who-paid-whom, chained several hops), which a graph database (e.g. Neo4j) answers naturally; a column-family store has no efficient way to follow such links without scanning everything.

## Part F: Design in practice
If you need to restart Part F, run `load_design(session)` in a cell:
```python
from load_cassandra import load_design
load_design(session)
```

### Question 16

In [21]:
# (a)
rows_per_day = 24 * 60 * 60 // 2
rows_per_year = rows_per_day * 365
print([rows_per_day, rows_per_year])

# (b)
run("""
CREATE TABLE vitals_by_patient_day (
    patient_id text,
    day date,
    reading_time timestamp,
    heart_rate int,
    spo2 int,
    PRIMARY KEY ((patient_id, day), reading_time)
) WITH CLUSTERING ORDER BY (reading_time DESC)
""")

# (c)
rows = run("SELECT reading_time, heart_rate, spo2 FROM vitals_by_patient "
           "WHERE patient_id = 'P-40232'")
for reading_time, heart_rate, spo2 in rows:
    run(f"INSERT INTO vitals_by_patient_day "
        f"(patient_id, day, reading_time, heart_rate, spo2) "
        f"VALUES ('P-40232', '2026-09-14', '{reading_time}+0000', {heart_rate}, {spo2})")

run("INSERT INTO vitals_by_patient_day (patient_id, day, reading_time, heart_rate, spo2) "
    "VALUES ('P-40232', '2026-09-15', '2026-09-15 00:00:00+0000', 86, 95)")
run("INSERT INTO vitals_by_patient_day (patient_id, day, reading_time, heart_rate, spo2) "
    "VALUES ('P-40232', '2026-09-15', '2026-09-15 00:00:02+0000', 85, 96)")

print(run("SELECT COUNT(*) FROM vitals_by_patient_day "
           "WHERE patient_id = 'P-40232' AND day = '2026-09-14'"))
print(run("SELECT COUNT(*) FROM vitals_by_patient_day "
           "WHERE patient_id = 'P-40232' AND day = '2026-09-15'"))

# (d)
try:
    run("SELECT COUNT(*) FROM vitals_by_patient_day WHERE patient_id = 'P-40232'")
except Exception as error:
    print(type(error).__name__)


[43200, 15768000]
[[10]]
[[2]]
InvalidRequest


### Question 17

In [22]:
# (a)
run("""
CREATE TABLE statement_by_account (
    account_id text,
    owner text STATIC,
    branch text STATIC,
    entry_time timestamp,
    amount int,
    description text,
    PRIMARY KEY ((account_id), entry_time)
) WITH CLUSTERING ORDER BY (entry_time DESC)
""")

# (b)
run("INSERT INTO statement_by_account (account_id, owner, branch) "
    "VALUES ('acct-771', 'Sara Ahmed', 'Mall Road')")

rows = run("SELECT entry_time, amount, description FROM ledger_by_account "
           "WHERE account_id = 'acct-771'")
for entry_time, amount, description in rows:
    run(f"INSERT INTO statement_by_account (account_id, entry_time, amount, description) "
        f"VALUES ('acct-771', '{entry_time}+0000', {amount}, '{description}')")

print(run("SELECT entry_time, owner, amount FROM statement_by_account "
           "WHERE account_id = 'acct-771'"))

# (c)
run("UPDATE statement_by_account SET branch = 'Gulberg' WHERE account_id = 'acct-771'")

print(run("SELECT entry_time, branch FROM statement_by_account "
           "WHERE account_id = 'acct-771' LIMIT 2"))


[['2026-09-14 15:20:00', 'Sara Ahmed', -800], ['2026-09-14 13:45:00', 'Sara Ahmed', 35000], ['2026-09-14 11:30:00', 'Sara Ahmed', -1200], ['2026-09-14 09:14:00', 'Sara Ahmed', -2500], ['2026-09-14 08:55:00', 'Sara Ahmed', 120000]]
[['2026-09-14 15:20:00', 'Gulberg'], ['2026-09-14 13:45:00', 'Gulberg']]


### Question 18

In [23]:
# (a)
run("""
CREATE TABLE vitals_recent (
    patient_id text,
    reading_time timestamp,
    heart_rate int,
    spo2 int,
    PRIMARY KEY ((patient_id), reading_time)
) WITH CLUSTERING ORDER BY (reading_time DESC) AND default_time_to_live = 604800
""")

print(run("SELECT default_time_to_live FROM system_schema.tables "
           "WHERE keyspace_name = 'hospital' "
           "AND table_name = 'vitals_recent'"))

# (b)
import time

run("INSERT INTO vitals_recent (patient_id, reading_time, heart_rate, spo2) "
    "VALUES ('P-40232', '2026-09-14 09:00:20+0000', 92, 86) USING TTL 20")
print(run("SELECT COUNT(*) FROM vitals_recent WHERE patient_id = 'P-40232'"))

time.sleep(25)
print(run("SELECT COUNT(*) FROM vitals_recent WHERE patient_id = 'P-40232'"))


[[604800]]
[[1]]
[[0]]


### Question 19

In [24]:
# (a)
def record_reading(patient_id, ward, time_text, heart_rate, spo2):
    ts = f"{time_text}+0000"
    statements = [
        f"INSERT INTO vitals_by_patient (patient_id, reading_time, heart_rate, spo2) "
        f"VALUES ('{patient_id}', '{ts}', {heart_rate}, {spo2})"
    ]
    if spo2 < 90:
        statements.append(
            f"INSERT INTO low_spo2_by_ward (ward, reading_time, patient_id, spo2) "
            f"VALUES ('{ward}', '{ts}', '{patient_id}', {spo2})"
        )
    batch = "BEGIN BATCH\n" + ";\n".join(statements) + ";\nAPPLY BATCH"
    run(batch)

# (b)
record_reading('P-40232', 'ICU-A', '2026-09-14 09:00:20', 92, 86)
record_reading('P-40232', 'ICU-A', '2026-09-14 09:00:22', 90, 91)
record_reading('P-40233', 'ICU-B', '2026-09-14 09:00:20', 68, 88)

print(run("SELECT reading_time, patient_id, spo2 FROM low_spo2_by_ward WHERE ward = 'ICU-A'"))
print(run("SELECT reading_time, patient_id, spo2 FROM low_spo2_by_ward WHERE ward = 'ICU-B'"))
print(run("SELECT COUNT(*) FROM vitals_by_patient WHERE patient_id = 'P-40232'"))


[['2026-09-14 09:00:20', 'P-40232', 86], ['2026-09-14 09:00:10', 'P-40232', 88], ['2026-09-14 09:00:08', 'P-40232', 89]]
[['2026-09-14 09:00:20', 'P-40233', 88], ['2026-09-14 09:00:14', 'P-40233', 87], ['2026-09-14 09:00:12', 'P-40233', 89]]
[[12]]


### Question 20

In [25]:
# (a)
run("""
CREATE TABLE alert_counts_by_day (
    day date,
    ward text,
    alerts counter,
    PRIMARY KEY ((day), ward)
) WITH CLUSTERING ORDER BY (ward ASC)
""")

# (b) -- run this cell only once; use TRUNCATE alert_counts_by_day and re-run if needed
for ward in ('ICU-A', 'ICU-B'):
    rows = run(f"SELECT reading_time FROM low_spo2_by_ward WHERE ward = '{ward}'")
    for _ in rows:
        run(f"UPDATE alert_counts_by_day SET alerts = alerts + 1 "
            f"WHERE day = '2026-09-14' AND ward = '{ward}'")

print(run("SELECT ward, alerts FROM alert_counts_by_day WHERE day = '2026-09-14'"))

# (c)
try:
    run("""
    BEGIN BATCH
    UPDATE alert_counts_by_day SET alerts = alerts + 1 WHERE day = '2026-09-14' AND ward = 'ICU-A';
    INSERT INTO vitals_by_patient (patient_id, reading_time, heart_rate, spo2)
        VALUES ('P-40232', '2026-09-14 09:00:24+0000', 88, 89);
    APPLY BATCH
    """)
except Exception as error:
    print(type(error).__name__)


[['ICU-A', 3], ['ICU-B', 3]]
InvalidRequest


### Question 21

In [26]:
# (a)
run("""
CREATE TABLE ledger_by_account_month (
    account_id text,
    month text,
    entry_time timestamp,
    amount int,
    description text,
    PRIMARY KEY ((account_id, month), entry_time)
) WITH CLUSTERING ORDER BY (entry_time DESC)
""")

run("""
CREATE TABLE accounts_by_customer (
    customer_id text,
    account_id text,
    account_type text,
    PRIMARY KEY ((customer_id), account_id)
)
""")

print(run("SELECT column_name, kind, clustering_order "
           "FROM system_schema.columns "
           "WHERE keyspace_name = 'hospital' "
           "AND table_name = 'ledger_by_account_month'"))

# (b)
for account in ('acct-771', 'acct-772'):
    rows = run(f"SELECT entry_time, amount, description FROM ledger_by_account "
               f"WHERE account_id = '{account}'")
    for entry_time, amount, description in rows:
        month = entry_time[:7]
        run(f"INSERT INTO ledger_by_account_month "
            f"(account_id, month, entry_time, amount, description) "
            f"VALUES ('{account}', '{month}', '{entry_time}+0000', {amount}, '{description}')")

october_entries = [
    ('2026-10-01 09:00:00', 35000, 'Salary'),
    ('2026-10-03 18:20:00', -4200, 'Groceries'),
    ('2026-10-05 12:00:00', -900, 'Bill payment'),
]
for entry_time, amount, description in october_entries:
    run(f"INSERT INTO ledger_by_account_month "
        f"(account_id, month, entry_time, amount, description) "
        f"VALUES ('acct-771', '2026-10', '{entry_time}+0000', {amount}, '{description}')")

print(run("SELECT entry_time, amount, description FROM ledger_by_account_month "
           "WHERE account_id = 'acct-771' AND month = '2026-10'"))
print(run("SELECT SUM(amount) FROM ledger_by_account_month "
           "WHERE account_id = 'acct-771' AND month = '2026-09'"))
print(run("SELECT SUM(amount) FROM ledger_by_account_month "
           "WHERE account_id = 'acct-771' AND month = '2026-10'"))

# (c)
run("INSERT INTO accounts_by_customer (customer_id, account_id, account_type) "
    "VALUES ('C-1001', 'acct-771', 'savings')")
run("INSERT INTO accounts_by_customer (customer_id, account_id, account_type) "
    "VALUES ('C-1001', 'acct-773', 'current')")
run("INSERT INTO accounts_by_customer (customer_id, account_id, account_type) "
    "VALUES ('C-1002', 'acct-772', 'savings')")

print(run("SELECT account_id, account_type FROM accounts_by_customer WHERE customer_id = 'C-1001'"))


[['account_id', 'partition_key', 'none'], ['amount', 'regular', 'none'], ['description', 'regular', 'none'], ['entry_time', 'clustering', 'desc'], ['month', 'partition_key', 'none']]
[['2026-10-05 12:00:00', -900, 'Bill payment'], ['2026-10-03 18:20:00', -4200, 'Groceries'], ['2026-10-01 09:00:00', 35000, 'Salary']]
[[150500]]
[[29900]]
[['acct-771', 'savings'], ['acct-773', 'current']]


### Question 22

In [27]:
# (a)
run("""
CREATE TABLE alerts_v1 (
    ward text, reading_time timestamp, patient_id text, spo2 int,
    PRIMARY KEY ((ward), reading_time))
""")

run("INSERT INTO alerts_v1 (ward, reading_time, patient_id, spo2) "
    "VALUES ('ICU-C', '2026-09-14 10:00:00+0000', 'P-40240', 88)")
run("INSERT INTO alerts_v1 (ward, reading_time, patient_id, spo2) "
    "VALUES ('ICU-C', '2026-09-14 10:00:00+0000', 'P-40241', 86)")

print(run("SELECT COUNT(*) FROM alerts_v1 WHERE ward = 'ICU-C'"))
print(run("SELECT patient_id, spo2 FROM alerts_v1 WHERE ward = 'ICU-C'"))

# (b)
run("""
CREATE TABLE vitals_by_time (
    reading_time timestamp, patient_id text, heart_rate int, spo2 int,
    PRIMARY KEY ((reading_time), patient_id))
""")

rows = run("SELECT reading_time, heart_rate, spo2 FROM vitals_by_patient "
           "WHERE patient_id = 'P-40233'")
for reading_time, heart_rate, spo2 in rows:
    run(f"INSERT INTO vitals_by_time (reading_time, patient_id, heart_rate, spo2) "
        f"VALUES ('{reading_time}+0000', 'P-40233', {heart_rate}, {spo2})")

try:
    run("SELECT reading_time, heart_rate, spo2 FROM vitals_by_time "
        "WHERE patient_id = 'P-40233' LIMIT 1")
except Exception as error:
    print(type(error).__name__)


[[1]]
[['P-40241', 86]]
InvalidRequest


### Question 22 (c)
Written answer (no code) — edit this cell with your own wording:

**alerts_v1:** The primary key is `((ward), reading_time)` — `patient_id` is not part of the key at all, so two different patients alerting in the same ward at the same `reading_time` land on the *same row* and the second insert silently overwrites the first (upsert). That's why only 1 row exists instead of 2.
Corrected key: `PRIMARY KEY ((ward), reading_time, patient_id)`.

**vitals_by_time:** The partition key is `reading_time`, and `patient_id` is only a clustering column. Cassandra can only query efficiently by partition key, so `WHERE patient_id = ...` with no `reading_time` requires a full scan and is refused (`InvalidRequest`) unless `ALLOW FILTERING` is added — which doesn't scale.
Corrected key: `PRIMARY KEY ((patient_id), reading_time)` (i.e. patient should be the partition key, since "latest reading of one patient" is the actual query).

## What to submit
- Export/print this notebook (or paste code + outputs) into one PDF or Word file named `RollNo_Name_Cassandra`.
- For Questions 1, 2, 4, 6, 8, 10, 13, 14, and 16–22, also include a screenshot of the output.
- For Question 15 and Question 22(c), give the written answer instead of a one-line answer (already drafted above — edit into your own words).